In [1]:
import requests
import zipfile
import io

In [2]:
url='http://mattmahoney.net/dc/text8.zip'
r=requests.get(url)
f=zipfile.ZipFile(io.BytesIO(r.content))
text=f.read('text8').decode('utf-8')

In [3]:
tokens=text.split()

In [4]:
len(tokens)

17005207

In [5]:
from collections import Counter
counter=Counter(tokens)
min_freq=5
vocab={}
for word,freq in counter.items():
    if freq>=min_freq:
        vocab[word]=len(vocab)

In [6]:
len(vocab)

71290

In [7]:
encoded=[vocab[token] for token in tokens if token in vocab]

In [8]:
import random
central_word=[]
pairs=[]
neg_words=1
for i,central_word in enumerate(encoded):
    contextual=[]
    for j in range(-2,3):
        if (j!=0 and 0<=i+j<len(encoded)):
            contextual_word=encoded[i+j]
            pairs.append((central_word,contextual_word,1))
            contextual.append(contextual_word)
    neg_count=0
    while (neg_count<neg_words):
        neg=random.randint(0,len(vocab)-1)
        if neg not in contextual:
            pairs.append((central_word,neg,0))
            neg_count+=1
            

In [9]:
import torch
torch.save({'vocab':vocab},'vocab_sgns.pth')

In [10]:
class sgns_dataset(torch.utils.data.Dataset):
    def __init__(self,pairs):
        self.pairs=pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,index):
        central,contextual,label=self.pairs[index]
        return torch.tensor(central),torch.tensor(contextual),torch.tensor(label,dtype=torch.float32)

In [11]:
dataset=sgns_dataset(pairs)

In [12]:
loader=torch.utils.data.DataLoader(dataset,batch_size=1024,shuffle=True)

In [13]:
import torch.nn as nn
class SGNSModel(torch.nn.Module):
    def __init__(self,vocab_size,output_dim):
        super(SGNSModel,self).__init__()
        self.W_hidden=nn.Parameter(torch.randn(vocab_size,output_dim)*0.01)
        self.W_out=nn.Parameter(torch.randn(vocab_size,output_dim)*0.01)
    def forward(self,central_ids,contextual_ids):
        v=self.W_hidden[central_ids]
        u=self.W_out[contextual_ids]
        score=torch.sum(v*u,dim=1)
        #prob=nn.Sigmoid(score)
        return score
        

In [14]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [15]:
model=SGNSModel(len(vocab),100).to(device)

In [16]:
optimizer=torch.optim.Adam(model.parameters())
loss_fn=nn.BCEWithLogitsLoss()

In [17]:
epochs=5
from tqdm import tqdm
for epoch in range(epochs):
    loss=0
    model.train()
    loop=tqdm(loader,desc=f'Epoch {epoch+1}/{epochs}', leave=False)
    for (central_ids,contextual_ids,y_batch) in loop:
        optimizer.zero_grad()
        central_ids=central_ids.to(device)
        contextual_ids=contextual_ids.to(device)
        y_batch=y_batch.to(device)
        y_pred=model(central_ids,contextual_ids)
        batch_loss=loss_fn(y_pred,y_batch)
        batch_loss.backward()
        optimizer.step()
        loop.set_postfix(loss=batch_loss.item())
        loss+=batch_loss.item()
    print(f'Training loss over epoch {epoch+1} is {loss/len(loader)}')
    torch.save({'model_state_dict':model.state_dict(),
              'optimizer_state_dict':optimizer.state_dict(),
              'epoch':epoch+1,
               'loss':loss},f'checkpoint_{epoch}.pth'
              )        

Training loss over epoch 1 is 0.21225876194176965


Training loss over epoch 2 is 0.18478886891174004


Training loss over epoch 3 is 0.16647583422111692


Training loss over epoch 4 is 0.15469531237352585


Training loss over epoch 5 is 0.14687254190766205


In [18]:
embeddings1=model.W_hidden
embeddings2=model.W_out
embeddings=(embeddings1+embeddings2)/2
print(embeddings.shape)

torch.Size([71290, 100])


In [19]:
torch.save(embeddings1,'embeddings_SGNS_hidden_5_epochs.pth')
torch.save(embeddings2,'embeddings_SGNS_output_5_epochs.pth')

In [20]:
import pandas as pd
df=pd.read_csv('wordsim353.csv')

In [21]:
import torch.nn.functional as F
sim=[]
for i in range(len(df)):
    word1=df['Word_1'][i]
    word2=df['Word_2'][i]
    if word1 in vocab and word2 in vocab:
        vec1=embeddings[vocab[word1]].unsqueeze(0)
        vec2=embeddings[vocab[word2]].unsqueeze(0)
        sim.append(F.cosine_similarity(vec1,vec2,dim=1).item())
    else:
        df=df.drop([i])
    

In [22]:
df['model_score']=sim
import scipy
print(scipy.stats.spearmanr(df['score'],df['model_score']))

SignificanceResult(statistic=0.20715343942695785, pvalue=0.0001404578303079576)
